# 10 报告模块 (report) 完整示例

覆盖 ExcelWriter / FeatureAnalyzer / feature_bin_stats / auto_model_report / ruleset_report / SwapAnalyzer。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os, tempfile
import hscredit as hsc
from hscredit import germancredit, init_setting
from hscredit import (
    ExcelWriter, dataframe2excel, feature_bin_stats, FeatureAnalyzer,
    ruleset_report, auto_model_report, compare_models,
    OptimalBinning, WOEEncoder, LogisticRegression, ScoreCard,
)
from sklearn.model_selection import train_test_split
init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
woe = WOEEncoder(max_n_bins=5)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
lr = LogisticRegression(max_iter=1000, calculate_stats=True)
lr.fit(X_woe_tr, y_tr)
y_prob_te = lr.predict_proba(X_woe_te)[:,1]
print('准备完成')

## 1. feature_bin_stats — 特征分箱统计

In [ ]:
# 单特征分箱统计表
bin_stats = feature_bin_stats(df, feature='duration.in.month', target=target, n_bins=6)
print(type(bin_stats))
bin_stats

## 2. FeatureAnalyzer — 多特征分析

In [ ]:
analyzer = FeatureAnalyzer(target=target, n_bins=5, method='mdlp')
analyzer.fit(df[num_feats[:5] + [target]])

print('IV汇总:')
print(analyzer.iv_summary_)

print('\n分箱统计(第一个特征):')
analyzer.bin_stats_[num_feats[0]].head()

## 3. ExcelWriter — 写入 Excel 报告

In [ ]:
tmpdir = tempfile.mkdtemp()
out_path = os.path.join(tmpdir, 'report.xlsx')

with ExcelWriter(out_path) as writer:
    # 写入多个sheet
    writer.write_dataframe(df.head(50), sheet_name='原始数据样例')
    writer.write_dataframe(analyzer.iv_summary_, sheet_name='IV汇总')
    for feat in num_feats[:3]:
        writer.write_dataframe(analyzer.bin_stats_[feat], sheet_name=f'分箱_{feat[:10]}')

print(f'Excel已保存: {out_path}')
print('文件大小:', os.path.getsize(out_path), 'bytes')

## 4. dataframe2excel — 快捷写入

In [ ]:
out2 = os.path.join(tmpdir, 'quick.xlsx')
dataframe2excel({'IV汇总': analyzer.iv_summary_, '原始数据': df.head(20)}, out2)
print('快捷Excel已保存:', out2)

## 5. auto_model_report — 自动模型评估报告

In [ ]:
report = auto_model_report(
    model=lr,
    X_train=X_woe_tr, y_train=y_tr,
    X_test=X_woe_te,  y_test=y_te,
    model_name='LogisticRegression',
)
print(type(report))
for k, v in report.items():
    print(f'  {k}: {v}')

## 6. compare_models — 多模型对比

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_woe_tr, y_tr)

cmp = compare_models(
    models={'LR': lr, 'RF': rf},
    X_test=X_woe_te, y_test=y_te,
)
cmp

## 7. ruleset_report — 规则集报告

In [ ]:
rules = [
    '`duration.in.month` > 36',
    '`credit.amount` > 5000',
    '`age.in.years` < 25',
]
rule_report = ruleset_report(df, rules=rules, target=target)
rule_report

## 8. SwapAnalyzer — 规则置换风险分析

In [ ]:
from hscredit import (SwapAnalyzer, SwapRiskConfig, create_swap_dataset,
    create_swap_dataset_from_rules, swap_analysis_report, SwapType)
np.random.seed(42)
df_swap = df.copy()
df_swap['old_decision'] = (lr.predict_proba(X_woe_te if len(df_swap)==len(X_woe_te) else woe.transform(X[num_feats]))[:,1] > 0.3).astype(int)
df_swap['new_score'] = lr.predict_proba(woe.transform(X[num_feats]))[:,1]
df_swap['new_decision'] = (df_swap['new_score'] > 0.25).astype(int)
swap_ds = create_swap_dataset(df_swap, old_decision='old_decision', new_decision='new_decision', target=target)
print('置换数据集:')
swap_ds.head()

In [ ]:
config = SwapRiskConfig(psi_threshold=0.2, bad_rate_lift_threshold=1.5)
analyzer_swap = SwapAnalyzer(config=config)
result = analyzer_swap.analyze(swap_ds, target=target)
print(result.summary())